In [1]:
import pandas as pd

df = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/batcher_data.txt', '\t', dtype=str, encoding='latin-1')

/home/erik/anaconda3/envs/openmmlab/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3444: FutureWarning: In a future version of pandas all arguments of read_csv except for the argument 'filepath_or_buffer' will be keyword-only
  exec(code_obj, self.user_global_ns, self.user_ns)


In [2]:
batches = df.groupby('batch')

In [3]:
spelling_dict = {}
list_of_batches = df.batch.unique()

for batch in list_of_batches:
    batch_dict = {}
    
    df_batch = batches.get_group(batch)
    batch_dict['trakt'] = df_batch.GTRAKT.unique()
    batch_dict['block'] = df_batch.GBLOCK.unique()
    batch_dict['enhet'] = df_batch.GENHET.unique()

    spelling_dict[batch] = batch_dict




In [1]:
#skriv om jaro som funktion för att undvika duplicated code
#continue på den andra exception också kanske, hur många såna är det förresten
#gå efter mallen i paddan och få ordentlig statistik på hur mycket vi rättar upp

import Levenshtein

def jaro(allowed, pred):
    jaro_list = []
    
    for inst in allowed:
        val = Levenshtein.ratio(pred, inst)
        jaro_list.append((inst, val))

    jaro_list = sorted(jaro_list, key=lambda x: x[1], reverse=True)

    return jaro_list

def false_positives():
    pass


with open('/home/erik/Riksarkivet/Projects/fastighet/output/eval_pred_million_500.txt', 'r') as f, open('/home/erik/Riksarkivet/Projects/fastighet/output/error_list_2_it.txt', 'w') as e:
    preds_gts = f.readlines()
    preds_gts = preds_gts[1:]

    total_ex = 0

    hits_block_enhet = 0
    hits = 0
    misses = 0
    hits_before_jaro = 0

    for pred_gt in preds_gts:

        jaro_trakt = []
        jaro_block = []
        jaro_enhet = []
        
        batch_pnr = pred_gt.split('|')[0].split('.')[0]
        batch = batch_pnr.split('_')[0]

        pred_whole = pred_gt.split('|')[2].strip()
        
        try:
            trakt_pred = pred_whole.split(';')[0]
            block_pred = pred_whole.split(';')[1]
            enhet_pred = pred_whole.split(';')[2]

            gt_whole = pred_gt.split('|')[1].lower()

            trakt_gt = gt_whole.split(';')[0]
            block_gt = gt_whole.split(';')[1]
            enhet_gt = gt_whole.split(';')[2]

            trakt_allowed = [x.lower() for x in spelling_dict[batch]['trakt']]
            block_allowed = [x.lower() for x in spelling_dict[batch]['block']]
            enhet_allowed = [x.lower() for x in spelling_dict[batch]['enhet']]
            total_ex += 1

        except:
            continue

        jaro_trakt = jaro(trakt_allowed, trakt_pred)
        jaro_block = jaro(block_allowed, block_pred)
        jaro_enhet = jaro(enhet_allowed, enhet_pred)

        sev_jaro_trakt = [x[0] for x in jaro_trakt[0:1]]
        sev_jaro_block = [x[0] for x in jaro_block[0:1]]
        sev_jaro_enhet = [x[0] for x in jaro_enhet[0:1]]

        #if jaro_trakt[0][0] == trakt_gt and jaro_block[0][0] == block_gt and jaro_enhet[0][0] == enhet_gt:
            #hits += 1
        
        
        if trakt_gt in sev_jaro_trakt and block_gt in sev_jaro_block and enhet_gt in sev_jaro_enhet:
            hits += 1

        elif trakt_pred == '*' and jaro_block[0][0] == block_gt and jaro_enhet[0][0] == enhet_gt:
            hits += 1

        elif trakt_pred == trakt_gt and block_pred == block_gt and enhet_pred == enhet_gt:
            hits += 1  

        elif jaro_trakt[0][0] == trakt_gt and block_pred == block_gt and enhet_pred == enhet_gt:
            hits += 1  

        #elif jaro_block[0][0] == block_gt and jaro_enhet[0][0] == enhet_gt or block_pred == block_gt and enhet_pred == enhet_gt:
        #    hits += 1

        else:
            e.write(batch + '\t' + batch_pnr + '\t' + gt_whole + '\t' + pred_whole + '\t' + jaro_trakt[0][0] + ';' + jaro_block[0][0] + ';' + jaro_enhet[0][0] + '\n')


        if trakt_pred == trakt_gt and block_pred == block_gt and enhet_pred == enhet_gt:
            hits_before_jaro += 1

    print(str(hits_before_jaro) + ':' + str(hits) +  ':' + str(total_ex))

        




0:0:0


In [14]:
val = Levenshtein.jaro_winkler('*', '*')
print(val)

1.0


In [31]:
print(spelling_dict['10009439']['block'])

['ABBORREN' 'STG' 'ADVOKATEN' 'ALBATROSSEN' 'ALEN' 'ALKAN' 'ALMEN' 'ANDEN'
 'ANKAN' 'ARKITEKTEN' 'ASPEN' 'BAGAREN' 'BECKASINEN' 'DUNDRET' 'BERGUVEN'
 'BISKOPEN' 'BJÖRKEN' 'BLÅKLINTEN' 'SOTHÖNAN' 'SVANEN' 'TÄRNAN' 'TRASTEN'
 'BLÅKLOCKAN' 'TRANAN' 'TALGOXEN' 'SKATAN' 'TJÄDERN' 'BLÅSIPPAN' 'TUPPEN'
 'BOFINKEN' 'BOKEN' 'BORRET' 'RÖNNEN' 'BRAXEN' 'BYÅLDERMANNEN'
 'BÖRSTINGEN' 'CEDERN' 'DALDOCKAN' 'DANSAREN' 'EJDERN' 'DIAKONEN'
 'DOMAREN' 'DOMHERREN' 'DOPPINGEN' 'DRILLSNÄPPAN' 'DUVAN' 'EKEN' 'ENEN'
 'ENGELSMANNEN' 'EPEDEMISJUKHUSET' 'FALKEN' 'FASANEN' 'KASTANJEN' 'ÄRLAN'
 'FJÄLLSIPPAN' 'VIPAN' 'FJÄRDINGSMANNEN' 'FLAMINGON' 'FLOTTAREN'
 'FLUNDRAN' 'FORELLEN' 'FÖRMANNEN' 'HERMELINEN' 'HJORTEN' 'GAMEN'
 'GAMLA SKOLAN' 'GLADAN' 'GOJAN' 'GRANEN' 'GRIPEN' 'GRÅSPARVEN'
 'GRÅSUGGAN' 'GRÅTRUTEN' 'GRÖNSISKAN' 'GULLVIVAN' 'GÄDDAN' 'PALMEN'
 'POPELN' 'MÖRTEN' 'HJÄRPEN' 'PIGGVAREN' 'KRÄFTAN' 'HAJEN' 'HACKSPETTEN'
 'CHAUFFÖREN' 'LAXEN' 'RUDAN' 'RÖDSPOTTAN' 'SILLEN' 'SPIGGEN' 'ÅLEN'
 'ZEBRAN' 'PRÄSTEN' 'VR

In [22]:
import Levenshtein

list_districts = spelling_dict['10032602']['block']


for dis in list_districts:
    val = Levenshtein.jaro_winkler('KÖPMANNEN', dis)
    if val > 0:
        print(dis + ': ' + str(val))
        

#10011222_00000037.tif|HÖRNETT;2;137|horna;2;137
#10032602_00000269.tif|*;KÖPMANNEN VÄSTRA;15|*;köpmannen;15

6674
ÅKERN: 0.5407407407407407
HÄSTHAGEN: 0.4444444444444444
STORMEN: 0.4761904761904761
PILPIPAN: 0.5694444444444444
ÅVÅNGEN: 0.4761904761904761
HEMMANSÄGAREN: 0.6467236467236467
HENNING: 0.5873015873015873
HENRIETTA: 0.48148148148148145
HENRIK: 0.4259259259259259
HERBERT: 0.41798941798941797
HERDEN: 0.35185185185185186
HERMAN: 0.611111111111111
HERMELINEN: 0.5314814814814816
VULCAN: 0.5185185185185185
HERMES: 0.5185185185185185
PALLAS: 0.5185185185185185
LIBANON: 0.5873015873015873
HERREMANNEN: 0.6262626262626262
HERTIGEN: 0.32407407407407407
HILDA: 0.437037037037037
HILDEGARD: 0.3148148148148148
HILDING: 0.41798941798941797
HILLEBARDEN: 0.42424242424242414
HILLEBORG: 0.4074074074074075
HILLEVI: 0.41798941798941797
HILLTORP NORRA: 0.45502645502645506
HILLTORP SÖDRA: 0.3941798941798942
HINKEN: 0.5370370370370371
HIRDMANNEN: 0.7555555555555555
HJALMAR: 0.33597883597883604
HJORTEN: 0.33597883597883604
HOLGER: 0.4259259259259259
HOLLAND: 0.5026455026455027
HOLMFRID: 0.41203703703703703
H